# Prompting lab: what the model actually sees

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankitrgupta1/silicon-to-agents/blob/main/labs/prompting.ipynb)

Three mechanics behind every LLM app: chat templates, constrained decoding (how structured output is guaranteed) and the arithmetic of prompt caching and context budgets. Companion to [Prompting](../07-applications/prompting.md). Runs on CPU with no API key: a tiny stand-in model replaces a real one where needed.

In [ ]:
try:
    import s2a
except ImportError:
    %pip install -q git+https://github.com/ankitrgupta1/silicon-to-agents

from s2a.prompting import (END, render_chatml, greedy_decode, constrained_decode, allowed_next,
                           toy_scorer, prefill_tokens, remaining_context)
from s2a.kv_cache import LLAMA3_8B, kv_cache_bytes, human_bytes

## 1. A chat is one long string

The API takes a list of messages, but the model receives a single piece of text with special markers. It ends with an open assistant turn, and the model's only job is to continue it.

In [ ]:
messages = [
    {"role": "system", "content": "You classify product reviews. Answer with one word: positive, negative or neutral."},
    {"role": "user", "content": "Review: Arrived broken and support never replied."},
]
print(render_chatml(messages))

Few-shot examples are just more turns before the real question: the model continues the pattern.

In [ ]:
few_shot = [
    messages[0],
    {"role": "user", "content": "Review: Works exactly as described."},
    {"role": "assistant", "content": "positive"},
    {"role": "user", "content": "Review: It's fine, nothing special."},
    {"role": "assistant", "content": "neutral"},
    messages[1],
]
print(render_chatml(few_shot))

## 2. Asking nicely versus guaranteeing the format

A stand-in "model" (random scores, one character at a time) is asked to output a label. Free generation produces junk. **Constrained decoding** masks every character that can't lead to a valid label, so the output is always valid while still following the model's own preferences among the valid choices.

In [ ]:
VOCAB = list("abcdefghijklmnopqrstuvwxyz ") + [END]
LABELS = ["positive", "negative", "neutral"]

for seed in range(5):
    model = toy_scorer(VOCAB, seed)
    free = greedy_decode(model, VOCAB, max_len=12)
    print(f"model {seed}: free {free!r:<16} constrained {constrained_decode(model, VOCAB, LABELS)!r}")

print("\nafter 'ne', the only allowed next characters are:", [VOCAB[i] for i in allowed_next("ne", LABELS, VOCAB)])

The same idea works with multi-character tokens and JSON. Real structured-output features compile a JSON schema into these masks.

In [ ]:
import json
vocab = ['{"sentiment": "', "pos", "neg", "neu", "itive", "ative", "tral", '"}', "oops", END]
options = [json.dumps({"sentiment": s}) for s in LABELS]
for seed in range(3):
    out = constrained_decode(toy_scorer(vocab, seed), vocab, options)
    print(out, "->", json.loads(out))

## 3. Prompt caching

An agent sends the same system prompt and tool definitions on every request. If the server keeps their KV cache, it only processes the new part each time.

In [ ]:
prefix, new, requests = 6_000, 500, 1_000
without = prefill_tokens(prefix, new, requests, cached=False)
with_cache = prefill_tokens(prefix, new, requests, cached=True)
print(f"input tokens processed: {without:,} without caching, {with_cache:,} with ({without / with_cache:.1f}x fewer)")
print(f"the cached prefix occupies {human_bytes(kv_cache_bytes(LLAMA3_8B, prefix))} of GPU memory (Llama-3-8B shape, BF16)")

## 4. The context budget

Everything has to fit in one window: instructions, tools, conversation history, retrieved documents, and room for the answer.

In [ ]:
budget = dict(system=2_000, tools=6_000, history=30_000, reserved_output=4_000)
print(f"room left for documents in a 128k window: {remaining_context(128_000, **budget):,} tokens")
try:
    remaining_context(32_000, **budget)
except ValueError as e:
    print("in a 32k window:", e)

## Try it

- Add a fourth label, `"mixed"`. Which prefixes now have more than one allowed next character?
- Use `toy_scorer(VOCAB, 0, bias={"n": 5.0, "g": 5.0})`. Which label does constrained decoding pick, and why?
- Change the caching example so the prefix is only 200 tokens. When is caching barely worth it?
- Constrained decoding guarantees the *format*. What can it not guarantee about the answer?